

**Install Libraries**


In [141]:
%%capture
import warnings
warnings.filterwarnings('ignore')

# Force install the specific huggingface_hub version first to ensure it's not overridden by other dependencies
!pip install --upgrade --force-reinstall huggingface_hub==0.34.0

# Install all other dependencies, explicitly pinning conflicting versions
!pip install \
    datasets==4.0.0 \
    faiss-cpu \
    transformers==4.46.3 \
    tokenizers==0.20.3 \
    sentence-transformers==3.3.1 \
    keybert==0.8.5 \
    langchain \
    langchain-community \
    langchain-core==0.3.86 \
    langchain-huggingface \
    langchain-groq \
    packaging==25.0 \
    fsspec==2025.3.0

from transformers import pipeline
type(pipeline)
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)
type(summarizer)

# The ner pipeline also uses transformers and should be initialized after compatible installations
ner = pipeline(
    "ner",
    aggregation_strategy="simple"
)

from langchain_huggingface import HuggingFacePipeline
llm = HuggingFacePipeline(
    pipeline=summarizer
)

from langchain_core.tools import tool
from langchain import agents
from langchain.agents import AgentExecutor, create_react_agent
from langchain_groq import ChatGroq
from google.colab import userdata
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Define the LLM (ensure api_key is available)
api_key = userdata.get("GROQ_API_KEY")
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=api_key,
    temperature=0
)

@tool
def search_and_summarize(query: str, k: int = 5) -> str:
    """
    Search research papers from the FAISS database,
    retrieve the top-k most similar papers,
    summarize each paper using BART,
    and return the results as structured data.
    """

    query_embedding = model.encode([query])

    faiss.normalize_L2(query_embedding)


    D, I = index.search(query_embedding, k)

    papers = []

    for rank, (score, idx) in enumerate(zip(D[0], I[0]), start=1):

        paper = df.iloc[idx]

        summary = summarizer(
            paper["abstract"],
            max_length=120,
            min_length=40,
            do_sample=False
        )[0]["summary_text"]


        paper_data = {
            "rank": rank,
            "similarity": round(float(score), 4),
            "title": paper["title"],
            "abstract": paper["abstract"],
            "summary": summary
        }

        papers.append(paper_data)

    return papers

@tool
def extract_keywords(text: str, top_n: int = 5) -> str:
    """
    Extract the most important keywords from the given text
    using the KeyBERT model.
    """


    keywords = kw_model.extract_keywords(
        text,
        keyphrase_ngram_range=(1, 2),
        stop_words="english",
        top_n=top_n
    )

    result = "Top Keywords:\n\n"

    for rank, (keyword, score) in enumerate(keywords, start=1):

        result += (
            f"{rank}. {keyword} "
            f"(Relevance Score: {round(score, 4)})\n"
        )

    return result

@tool
def compare_papers(
    comparison_query: str
) -> str:
    """
    Compares two research papers based on their abstracts.
    The `comparison_query` string should contain the titles of the two papers to compare,
    separated by ' vs ' or ' and '. The tool will extract these titles,
    retrieve the closest matching papers from the FAISS database,
    and use the LLM to compare them.

    Example: 'Compare paper "Title A" vs paper "Title B"'
    """

    if ' vs ' in comparison_query:
        parts = comparison_query.split(' vs ', 1)
    elif ' and ' in comparison_query:
        parts = comparison_query.split(' and ', 1)
    else:
        raise ValueError("comparison_query must contain ' vs ' or ' and ' to separate paper titles.")

    paper1_title = parts[0].replace('paper "', '').replace('"', '').strip()
    paper2_title = parts[1].replace('paper "', '').replace('"', '').strip()

    embedding1 = model.encode([paper1_title])
    faiss.normalize_L2(embedding1)
    D1, I1 = index.search(embedding1, 1)
    first_paper = df.iloc[I1[0][0]]

    embedding2 = model.encode([paper2_title])
    faiss.normalize_L2(embedding2)
    D2, I2 = index.search(embedding2, 1)
    second_paper = df.iloc[I2[0][0]]

    comparison_prompt = f"""
Compare the following two research papers.

Paper 1

Title:
{first_paper['title']}

Abstract:
{first_paper['abstract']}


Paper 2

Title:
{second_paper['title']}

Abstract:
{second_paper['abstract']}


Compare them based on:

1. Research Objective

2. Methodology

3. Key Contributions

4. Advantages

5. Limitations

6. Applications

Present the comparison in a clear table.
"""
    response = llm.invoke(comparison_prompt)
    return response.content

tools = [
    search_and_summarize,
    extract_keywords,
    compare_papers
]

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful AI assistant. You have access to the following tools: {tools}\n"
            "You MUST use the following format:\n\n"
            "Question: the input question you must answer\n"
            "Thought: you should always think about what to do\n"
            "Action: the action to take, MUST be one of [{tool_names}]\n"
            "Action Input: the input to the action\n"
            "Observation: the result of the action\n"
            "... (this Thought/Action/Action Input/Observation can repeat N times)\n"
            "Thought: I have gathered all necessary information and am ready to provide a final answer.\n"
            "Final Answer: the final answer to the original input question\n\n"
            "Crucial Rule: Do NOT invent new actions. Only use the tools provided or output a Final Answer when your task is complete.\n"
            "Special Instruction: When using the 'compare_papers' tool, provide a single `comparison_query` string containing the two paper titles, separated by ' vs ' or ' and '. For example: `Action Input: comparison_query='Compare paper \"Title A\" vs paper \"Title B\"'` "
        ),
        ("human", "{input}\n{agent_scratchpad}"),
    ]
)

agent_runnable = create_react_agent(llm, tools, prompt)
agent = AgentExecutor(agent=agent_runnable, tools=tools, verbose=True)

print("Agent updated with new tools!")

In [2]:
import os
from google.colab import userdata

groq_api_key = userdata.get('GROQ_API_KEY')
os.environ["GROQ_API_KEY"] = groq_api_key

print("Groq API key set ho gayi!")

Groq API key set ho gayi!


In [3]:
import os
from google.colab import userdata
from huggingface_hub import login

# Secret se token lena
hf_token = userdata.get('HF_TOKEN')

# Login karna
login(token=hf_token)

print("Hugging Face login successful!")

Hugging Face login successful!


In [4]:
from datasets import load_dataset

**Load Dataset**

In [5]:
dataset = load_dataset(
    "CShorten/ML-ArXiv-Papers",
    split="train"
)

In [6]:
print(dataset)

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'],
    num_rows: 117592
})


In [7]:
dataset[0]

{'Unnamed: 0.1': 0,
 'Unnamed: 0': 0.0,
 'title': 'Learning from compressed observations',
 'abstract': '  The problem of statistical learning is to construct a predictor of a random\nvariable $Y$ as a function of a related random variable $X$ on the basis of an\ni.i.d. training sample from the joint distribution of $(X,Y)$. Allowable\npredictors are drawn from some specified class, and the goal is to approach\nasymptotically the performance (expected loss) of the best predictor in the\nclass. We consider the setting in which one has perfect observation of the\n$X$-part of the sample, while the $Y$-part has to be communicated at some\nfinite bit rate. The encoding of the $Y$-values is allowed to depend on the\n$X$-values. Under suitable regularity conditions on the admissible predictors,\nthe underlying family of probability distributions and the loss function, we\ngive an information-theoretic characterization of achievable predictor\nperformance in terms of conditional distortion-rat

**dataset into a Pandas DataFrame.**

In [8]:
import pandas as pd

df = pd.DataFrame(dataset)

df.head()

,Unnamed: 0.1,Unnamed: 0,title,abstract
0,0,0.0,Learning from compressed observations,The problem of statistical learning is to co...
1,1,1.0,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,2,2.0,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,3,3.0,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,4,4.0,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...


In [9]:
df.shape

(117592, 4)

In [10]:
df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'], dtype='object')

In [11]:
df = df[['title', 'abstract']]

In [12]:
df.head()

,title,abstract
0,Learning from compressed observations,The problem of statistical learning is to co...
1,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...


In [13]:
df = df.head(15000)

In [14]:
df.shape

(15000, 2)

In [15]:
df.isnull().sum()

,0
title,0
abstract,0


In [16]:
df["paper_text"] = df["title"] + " " + df["abstract"]

In [17]:
df[["paper_text"]].head()

,paper_text
0,Learning from compressed observations The pr...
1,Sensor Networks with Random Links: Topology De...
2,The on-line shortest path problem under partia...
3,A neural network approach to ordinal regressio...
4,Parametric Learning and Monte Carlo Optimizati...


In [18]:
type(df[["paper_text"]])

pandas.core.frame.DataFrame

In [19]:
df["paper_text"].head()

,paper_text
0,Learning from compressed observations The pr...
1,Sensor Networks with Random Links: Topology De...
2,The on-line shortest path problem under partia...
3,A neural network approach to ordinal regressio...
4,Parametric Learning and Monte Carlo Optimizati...


In [20]:
type(df["paper_text"])

pandas.core.series.Series

In [21]:
print(df["paper_text"].iloc[0])

Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random
variable $Y$ as a function of a related random variable $X$ on the basis of an
i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable
predictors are drawn from some specified class, and the goal is to approach
asymptotically the performance (expected loss) of the best predictor in the
class. We consider the setting in which one has perfect observation of the
$X$-part of the sample, while the $Y$-part has to be communicated at some
finite bit rate. The encoding of the $Y$-values is allowed to depend on the
$X$-values. Under suitable regularity conditions on the admissible predictors,
the underlying family of probability distributions and the loss function, we
give an information-theoretic characterization of achievable predictor
performance in terms of conditional distortion-rate functions. The ideas are
illustrated on the example of nonparametric regress

In [22]:
df["paper_text"] = df["paper_text"].str.replace("\n", " ", regex=False)
df["paper_text"] = df["paper_text"].str.strip()

In [23]:
print(df["paper_text"].iloc[0])

Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random variable $Y$ as a function of a related random variable $X$ on the basis of an i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable predictors are drawn from some specified class, and the goal is to approach asymptotically the performance (expected loss) of the best predictor in the class. We consider the setting in which one has perfect observation of the $X$-part of the sample, while the $Y$-part has to be communicated at some finite bit rate. The encoding of the $Y$-values is allowed to depend on the $X$-values. Under suitable regularity conditions on the admissible predictors, the underlying family of probability distributions and the loss function, we give an information-theoretic characterization of achievable predictor performance in terms of conditional distortion-rate functions. The ideas are illustrated on the example of nonparametric regress

**Sentence Transformers**

In [24]:
from sentence_transformers import SentenceTransformer

In [25]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [26]:
print(type(model))

<class 'sentence_transformers.SentenceTransformer.SentenceTransformer'>


In [27]:
sample_text = df["paper_text"].iloc[0]

embedding = model.encode(sample_text)

print(type(embedding))
print(embedding.shape)

<class 'numpy.ndarray'>
(384,)


In [28]:
embedding[:10]

array([-0.1315641 , -0.00678266, -0.00367612,  0.03265158,  0.11219642,
        0.01227267,  0.09816719, -0.0900523 ,  0.04231161, -0.01977348],
      dtype=float32)

In [29]:
sample_embeddings = model.encode(
    df["paper_text"].head(5).tolist()
)

print(type(sample_embeddings))
print(sample_embeddings.shape)

<class 'numpy.ndarray'>
(5, 384)


In [30]:
from sklearn.metrics.pairwise import cosine_similarity

In [31]:
similarity = cosine_similarity(
    sample_embeddings[0].reshape(1, -1),
    sample_embeddings[1].reshape(1, -1)
)

print(similarity)

[[0.36625272]]


In [32]:
similarity = cosine_similarity(
    sample_embeddings[0].reshape(1,-1),
    sample_embeddings[0].reshape(1,-1)
)

print(similarity)

[[1.0000001]]


In [33]:
for i in range(1,5):

    sim = cosine_similarity(
        sample_embeddings[0].reshape(1,-1),
        sample_embeddings[i].reshape(1,-1)
    )

    print(f"Paper 1 vs Paper {i+1}: {sim[0][0]:.4f}")

Paper 1 vs Paper 2: 0.3663
Paper 1 vs Paper 3: 0.3352
Paper 1 vs Paper 4: 0.1551
Paper 1 vs Paper 5: 0.3742


**Generate Full Embeddings**

In [34]:
import os
import numpy as np

if os.path.exists("paper_embeddings.npy"):

    print("Loading saved embeddings...")

    embeddings = np.load("paper_embeddings.npy")

else:

    print("Generating embeddings...")

    embeddings = model.encode(
        df["paper_text"].tolist(),
        batch_size=32,
        show_progress_bar=True
    )

    np.save("paper_embeddings.npy", embeddings)

    print("Embeddings saved successfully!")#embeddings = model.encode(
    #df["paper_text"].tolist(),
    #batch_size=32,
    #show_progress_bar=True
#)

Loading saved embeddings...


In [35]:
print(type(embeddings))
print(embeddings.shape)


<class 'numpy.ndarray'>
(15000, 384)


In [36]:
embeddings.nbytes / (1024*1024)

21.97265625

In [37]:
embeddings.dtype


dtype('float32')

In [38]:
import os
import faiss

if os.path.exists("paper_faiss.index"):

    print("Loading existing FAISS index...")

    index = faiss.read_index("paper_faiss.index")

else:

    print("Creating new FAISS index...")

    faiss.normalize_L2(embeddings)

    index = faiss.IndexFlatIP(384)

    index.add(embeddings)

    faiss.write_index(index, "paper_faiss.index")

    print("FAISS index saved successfully!")#faiss.normalize_L2(embeddings)

#index = faiss.IndexFlatIP(384)

#index.add(embeddings)

Loading existing FAISS index...


In [39]:
import numpy as np

np.linalg.norm(embeddings[0])

np.float32(1.0)

In [40]:
print(index.ntotal)

15000


In [41]:
query = "deep learning for medical image analysis"
query_embedding = model.encode([query])
query_embedding.shape


(1, 384)

In [42]:
faiss.normalize_L2(query_embedding)


np.linalg.norm(query_embedding[0])

np.float32(1.0)

In [43]:
D, I = index.search(query_embedding, 5)

print(D)
print(I)

[[0.6807244  0.67092204 0.65219975 0.62811744 0.61311525]]
[[10466 13730 11873 12691 11282]]


In [44]:
print(df.iloc[10466]["title"])

A Perspective on Deep Imaging


In [45]:
print(df.iloc[10466]["abstract"])

  The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance in clinical and
preclinical applications. To realize the full impact of machine learning on
medical imaging, major challenges must be addressed.



In [46]:
def search_papers(query, k=5):

    query_embedding = model.encode([query])

    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, k)

    return D, I

In [47]:
D, I = search_papers(
    "deep learning for medical image analysis"
)

print(D)
print(I)

[[0.6807244  0.67092204 0.65219975 0.62811744 0.61311525]]
[[10466 13730 11873 12691 11282]]


In [48]:
def search_papers(query, k=5):

    query_embedding = model.encode([query])

    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, k)

    for score, idx in zip(D[0], I[0]):

        print("=" * 100)

        print("Similarity:", round(float(score), 4))

        print("Title:", df.iloc[idx]["title"])

        print()

        print("Abstract:")

        print(df.iloc[idx]["abstract"][:500])

        print()

In [49]:
search_papers(
    "deep learning for medical image analysis"
)

Similarity: 0.6807
Title: A Perspective on Deep Imaging

Abstract:
  The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance

Similarity: 0.6709
Title: Convolutional Neural Networks for Medical Image Analysis: Full Training
  or Fine Tuning?

Abstract:
  Training a deep convolutional neural network (CNN) from scratch is difficult
because it requires a large amount of labeled training data and a great deal of
expertise to ensure proper convergence. A promising alternative is to fine-tune
a CNN that has been pre-trained using, for instance, a large set 

In [50]:
import transformers
import huggingface_hub

print("Transformers:", transformers.__version__)
print("Transformers Path:", transformers.__file__)

print("Hub:", huggingface_hub.__version__)
print("Hub Path:", huggingface_hub.__file__)

import transformers.utils as u

print(hasattr(u, "HUGGINGFACE_CO_RESOLVE_ENDPOINT"))

Transformers: 4.46.3
Transformers Path: /usr/local/lib/python3.12/dist-packages/transformers/__init__.py
Hub: 0.34.0
Hub Path: /usr/local/lib/python3.12/dist-packages/huggingface_hub/__init__.py
True


In [51]:
import transformers.utils as u

print(hasattr(u, "HUGGINGFACE_CO_RESOLVE_ENDPOINT"))

True


In [52]:
type(summarizer)

transformers.pipelines.text2text_generation.SummarizationPipeline

In [53]:
summary = summarizer(
    df.iloc[10466]["abstract"],
    max_length=120,
    min_length=40,
    do_sample=False
)

In [54]:
print(summary)

[{'summary_text': 'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.'}]


In [55]:
type(summary)

list

In [56]:
summary[0]

{'summary_text': 'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.'}

In [57]:
type(summary[0])

dict

In [58]:
summary[0]["summary_text"]

'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.'

In [59]:
print(summary[0]["summary_text"])

The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.


In [60]:
for score, idx in zip(D[0], I[0]):

    print("=" * 100)

    print("Similarity:", round(float(score), 4))

    print("Title:", df.iloc[idx]["title"])

    print()

    summary = summarizer(
        df.iloc[idx]["abstract"],
        max_length=120,
        min_length=40,
        do_sample=False
    )

    print("Summary:")

    print(summary[0]["summary_text"])

    print()

Similarity: 0.6807
Title: A Perspective on Deep Imaging

Summary:
The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.

Similarity: 0.6709
Title: Convolutional Neural Networks for Medical Image Analysis: Full Training
  or Fine Tuning?

Summary:
Training a deep convolutional neural network from scratch is difficult because it requires a large amount of labeled training data. A promising alternative is to fine-tune a CNN that has been pre-trained using, for instance, a large set of labeled natural images.

Similarity: 0.6522
Title: Classification of MRI data using Deep Learning and Gaussian
  Process-based Model Selection

Summary:
The classification of MRI images according to the anatomical field of vie

In [61]:
def summarize_text(text):

    summary = summarizer(
        text,
        max_length=120,
        min_length=40,
        do_sample=False
    )

    return summary[0]["summary_text"]

In [62]:
def search_and_summarize(query, k=5):

    query_embedding = model.encode([query])

    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, k)

    for score, idx in zip(D[0], I[0]):

        print("=" * 100)

        print("Similarity:", round(float(score), 4))

        print("Title:", df.iloc[idx]["title"])

        print()

        summary = summarizer(
            df.iloc[idx]["abstract"],
            max_length=120,
            min_length=40,
            do_sample=False
        )

        print("Summary:")

        print(summary[0]["summary_text"])

        print()

In [63]:
search_and_summarize(
    "Deep Learning in Medical Imaging",
    k=5
)

Similarity: 0.7355
Title: A Perspective on Deep Imaging

Summary:
The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.

Similarity: 0.6882
Title: Classification of MRI data using Deep Learning and Gaussian
  Process-based Model Selection

Summary:
The classification of MRI images according to the anatomical field of view is a necessary task to solve when faced with the increasing quantity of medical images. Using a common architecture (such as AlexNet) provides quite good results, but not sufficient for clinical use.

Similarity: 0.6532
Title: Convolutional Neural Networks for Medical Image Analysis: Full Training
  or Fine Tuning?

Summary:
Training a deep convolutional neural network from scratch is d

In [64]:
from keybert import KeyBERT

In [65]:
kw_model = KeyBERT(model)

In [66]:
type(kw_model)

keybert._model.KeyBERT

In [67]:
text = df.iloc[10466]["abstract"]

keywords = kw_model.extract_keywords(text)

In [68]:
print(type(keywords))

<class 'list'>


In [69]:
print(type(keywords[0]))

<class 'tuple'>


In [70]:
print(keywords[0])

('imaging', 0.4528)


In [71]:
keywords = kw_model.extract_keywords(
    text,
    keyphrase_ngram_range=(1,3),
    stop_words="english",
    top_n=5
)

print(keywords)

[('tomographic imaging deep', 0.6704), ('imaging deep learning', 0.6543), ('learning medical imaging', 0.6041), ('imaging deep', 0.5919), ('medical imaging', 0.5281)]


In [72]:
def search_and_summarize(query, k=5):

    query_embedding = model.encode([query])

    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, k)

    for score, idx in zip(D[0], I[0]):

        print("=" * 100)

        print("Similarity:", round(float(score), 4))

        print("Title:", df.iloc[idx]["title"])

        print()

        summary = summarizer(
            df.iloc[idx]["abstract"],
            max_length=120,
            min_length=40,
            do_sample=False
        )

        print("Summary:")

        print(summary[0]["summary_text"])

        print()

        keywords = kw_model.extract_keywords(
            df.iloc[idx]["abstract"],
            keyphrase_ngram_range=(1,3),
            stop_words="english",
            top_n=5
        )

        print("Keywords:")

        for keyword, score in keywords:
            print("-", keyword)

        print()

In [73]:
search_and_summarize(
    "Deep Learning in Medical Imaging",
    k=5
)

Similarity: 0.7355
Title: A Perspective on Deep Imaging

Summary:
The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.

Keywords:
- tomographic imaging deep
- imaging deep learning
- learning medical imaging
- imaging deep
- medical imaging

Similarity: 0.6882
Title: Classification of MRI data using Deep Learning and Gaussian
  Process-based Model Selection

Summary:
The classification of MRI images according to the anatomical field of view is a necessary task to solve when faced with the increasing quantity of medical images. Using a common architecture (such as AlexNet) provides quite good results, but not sufficient for clinical use.

Keywords:
- classification mri images
- classification mri
- mri d

**Hybrid NER Architecture**

In [74]:
text = """
ResNet50 was trained on ImageNet using PyTorch
at Stanford University.
"""
entities = ner(text)

print(entities)

[{'entity_group': 'ORG', 'score': np.float32(0.89313775), 'word': 'ResNet50', 'start': 1, 'end': 9}, {'entity_group': 'ORG', 'score': np.float32(0.82662463), 'word': 'ImageNet', 'start': 25, 'end': 33}, {'entity_group': 'ORG', 'score': np.float32(0.94511735), 'word': 'PyTorch', 'start': 40, 'end': 47}, {'entity_group': 'ORG', 'score': np.float32(0.93374425), 'word': 'Stanford University', 'start': 51, 'end': 70}]


In [75]:

from langchain_core.tools import tool
@tool
def search_papers(query: str) -> str:
    """
    Search research papers from the FAISS database
    and return the most relevant results.
    """

In [76]:
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x7a854bbdbc20>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7a854ba2c0e0>, model_name='llama-3.1-8b-instant', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [77]:
response = llm.invoke(
    "Deep Learning has become one of the most successful techniques for medical image reconstruction using convolutional neural networks."
)

In [78]:
response

AIMessage(content='Deep learning, particularly convolutional neural networks (CNNs), has revolutionized medical image reconstruction by providing state-of-the-art results in various applications. Here are some key aspects of deep learning in medical image reconstruction:\n\n1. **Image Denoising**: CNNs can effectively remove noise from medical images, improving the quality and accuracy of diagnoses.\n2. **Image Super-Resolution**: Deep learning models can enhance the resolution of low-resolution medical images, allowing for more detailed analysis and diagnosis.\n3. **Image Segmentation**: CNNs can automatically segment medical images, identifying specific structures or lesions, which is crucial for diagnosis and treatment planning.\n4. **Image Reconstruction**: Deep learning models can reconstruct medical images from incomplete or noisy data, such as MRI or CT scans, improving image quality and reducing radiation exposure.\n5. **Multi-Modal Fusion**: CNNs can fuse data from multiple im

In [79]:
print(type(response))

<class 'langchain_core.messages.ai.AIMessage'>


In [80]:
from langchain_groq import ChatGroq

In [81]:
from google.colab import userdata

In [82]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=api_key,
    temperature=0
)

In [83]:
response = llm.invoke("Hello, who are you?")

In [84]:
response

AIMessage(content="I'm an artificial intelligence model known as a large language model (LLM) or conversational AI. I'm a computer program designed to understand and respond to human language in a way that's natural and helpful.\n\nI don't have a personal identity or emotions like humans do, but I'm here to assist you with any questions, topics, or tasks you'd like to discuss. I can provide information, answer questions, generate text, and even engage in conversation.\n\nI'm constantly learning and improving my responses based on the data and interactions I receive, so feel free to ask me anything and see how I can help!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 126, 'prompt_tokens': 41, 'total_tokens': 167, 'completion_time': 0.1546447, 'completion_tokens_details': None, 'prompt_time': 0.003290976, 'prompt_tokens_details': None, 'queue_time': 0.00671925, 'total_time': 0.157935676}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_02

In [85]:
response.content

"I'm an artificial intelligence model known as a large language model (LLM) or conversational AI. I'm a computer program designed to understand and respond to human language in a way that's natural and helpful.\n\nI don't have a personal identity or emotions like humans do, but I'm here to assist you with any questions, topics, or tasks you'd like to discuss. I can provide information, answer questions, generate text, and even engage in conversation.\n\nI'm constantly learning and improving my responses based on the data and interactions I receive, so feel free to ask me anything and see how I can help!"

In [86]:
from langchain_core.tools import tool

In [87]:


@tool
def search_and_summarize(query: str, k: int = 5) -> str:
    """
    Search research papers from the FAISS database,
    retrieve the top-k most similar papers,
    summarize each paper using BART,
    and return the results.
    """


    query_embedding = model.encode([query])


    faiss.normalize_L2(query_embedding)


    D, I = index.search(query_embedding, k)


    result = ""

    for rank, (score, idx) in enumerate(zip(D[0], I[0]), start=1):

        paper = df.iloc[idx]


        summary = summarizer(
            paper["abstract"],
            max_length=120,
            min_length=40,
            do_sample=False
        )[0]["summary_text"]


        result += f"Rank: {rank}\n"
        result += f"Similarity Score: {round(float(score),4)}\n"
        result += f"Title: {paper['title']}\n\n"


        result += paper["abstract"] + "\n\n"


        result += summary + "\n\n"

    return result

In [88]:
tools = [search_and_summarize]

In [90]:
from langchain.agents import create_react_agent

In [92]:
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [117]:
response = agent.invoke(
    {
        "input": "Find the top 3 research papers on Vision Transformer."
    }
)



> Entering new AgentExecutor chain...
Thought: To find the top 3 research papers on Vision Transformer, I need to search the FAISS database for papers related to Vision Transformer and then summarize the top results.

Action: search_and_summarize
Action Input: query="Vision Transformer", k=3
[{'rank': 1, 'similarity': 0.358, 'title': 'Pareto-depth for Multiple-query Image Retrieval', 'abstract': '  Most content-based image retrieval systems consider either one single query,\nor multiple queries that include the same object or represent the same semantic\ninformation. In this paper we consider the content-based image retrieval\nproblem for multiple query images corresponding to different image semantics.\nWe propose a novel multiple-query information retrieval algorithm that combines\nthe Pareto front method (PFM) with efficient manifold ranking (EMR). We show\nthat our proposed algorithm outperforms state of the art multiple-query\nretrieval algorithms on real-world image databases. 

In [118]:
response

{'input': 'Find the top 3 research papers on Vision Transformer.',
 'output': "The top 3 research papers on Vision Transformer are:\n\n1. 'Pareto-depth for Multiple-query Image Retrieval'\n2. 'Building a Large-scale Multimodal Knowledge Base System for Answering Visual Queries'\n3. 'Thesis: Multiple Kernel Learning for Object Categorization'\n\nThese papers have been summarized as follows:\n\n1. 'Pareto-depth for Multiple-query Image Retrieval': Most content-based image retrieval systems consider either one single query, or multiple queries that include the same object or represent the same semantic information. We propose a novel multiple-query information retrieval algorithm that combines the Pareto front method (PFM) with efficient manifold ranking (EMR).\n2. 'Building a Large-scale Multimodal Knowledge Base System for Answering Visual Queries': Today's vision systems would still struggle to deal with visual queries that require a deeper reasoning. We propose a knowledge base (KB) f

In [119]:
print(type(response))

<class 'dict'>


In [120]:
print(response.keys())

dict_keys(['input', 'output'])


In [122]:
# The 'response' object from agent.invoke is a dictionary containing 'input' and 'output'.
# It does not have a 'messages' key in this context.
# If you want to access the final output of the agent, use response['output'].
# If you were expecting a list of messages, the AgentExecutor's invoke method doesn't return them directly in this structure.

# To see the available keys, you can run:
# print(response.keys())

# If you intended to get the length of the agent's final output string, you can do:
len(response["output"])

1520

In [124]:
# The 'response' object from agent.invoke contains 'input' and 'output' keys, not 'messages'.
# To access the agent's final output, you should use response['output'].
# If you intend to access messages in a different context, the structure might be different.

# Corrected code to access the agent's output:
response["output"]

"The top 3 research papers on Vision Transformer are:\n\n1. 'Pareto-depth for Multiple-query Image Retrieval'\n2. 'Building a Large-scale Multimodal Knowledge Base System for Answering Visual Queries'\n3. 'Thesis: Multiple Kernel Learning for Object Categorization'\n\nThese papers have been summarized as follows:\n\n1. 'Pareto-depth for Multiple-query Image Retrieval': Most content-based image retrieval systems consider either one single query, or multiple queries that include the same object or represent the same semantic information. We propose a novel multiple-query information retrieval algorithm that combines the Pareto front method (PFM) with efficient manifold ranking (EMR).\n2. 'Building a Large-scale Multimodal Knowledge Base System for Answering Visual Queries': Today's vision systems would still struggle to deal with visual queries that require a deeper reasoning. We propose a knowledge base (KB) framework to handle an assortment of visual queries. Building such a large-scal

In [126]:
# The 'response' object from agent.invoke contains 'input' and 'output' keys, not 'messages'.
# To access the agent's final output, you should use response['output'].
# If you intend to access messages in a different context, the structure might be different.

# Corrected code to access the agent's output:
print(response["output"])

The top 3 research papers on Vision Transformer are:

1. 'Pareto-depth for Multiple-query Image Retrieval'
2. 'Building a Large-scale Multimodal Knowledge Base System for Answering Visual Queries'
3. 'Thesis: Multiple Kernel Learning for Object Categorization'

These papers have been summarized as follows:

1. 'Pareto-depth for Multiple-query Image Retrieval': Most content-based image retrieval systems consider either one single query, or multiple queries that include the same object or represent the same semantic information. We propose a novel multiple-query information retrieval algorithm that combines the Pareto front method (PFM) with efficient manifold ranking (EMR).
2. 'Building a Large-scale Multimodal Knowledge Base System for Answering Visual Queries': Today's vision systems would still struggle to deal with visual queries that require a deeper reasoning. We propose a knowledge base (KB) framework to handle an assortment of visual queries. Building such a large-scale multimod

In [129]:
print(response['output'])

The top 3 research papers on Vision Transformer are:

1. 'Pareto-depth for Multiple-query Image Retrieval'
2. 'Building a Large-scale Multimodal Knowledge Base System for Answering Visual Queries'
3. 'Thesis: Multiple Kernel Learning for Object Categorization'

These papers have been summarized as follows:

1. 'Pareto-depth for Multiple-query Image Retrieval': Most content-based image retrieval systems consider either one single query, or multiple queries that include the same object or represent the same semantic information. We propose a novel multiple-query information retrieval algorithm that combines the Pareto front method (PFM) with efficient manifold ranking (EMR).
2. 'Building a Large-scale Multimodal Knowledge Base System for Answering Visual Queries': Today's vision systems would still struggle to deal with visual queries that require a deeper reasoning. We propose a knowledge base (KB) framework to handle an assortment of visual queries. Building such a large-scale multimod

In [132]:
print(response["output"])

The top 3 research papers on Vision Transformer are:

1. 'Pareto-depth for Multiple-query Image Retrieval'
2. 'Building a Large-scale Multimodal Knowledge Base System for Answering Visual Queries'
3. 'Thesis: Multiple Kernel Learning for Object Categorization'

These papers have been summarized as follows:

1. 'Pareto-depth for Multiple-query Image Retrieval': Most content-based image retrieval systems consider either one single query, or multiple queries that include the same object or represent the same semantic information. We propose a novel multiple-query information retrieval algorithm that combines the Pareto front method (PFM) with efficient manifold ranking (EMR).
2. 'Building a Large-scale Multimodal Knowledge Base System for Answering Visual Queries': Today's vision systems would still struggle to deal with visual queries that require a deeper reasoning. We propose a knowledge base (KB) framework to handle an assortment of visual queries. Building such a large-scale multimod

In [133]:
print(response["output"])

The top 3 research papers on Vision Transformer are:

1. 'Pareto-depth for Multiple-query Image Retrieval'
2. 'Building a Large-scale Multimodal Knowledge Base System for Answering Visual Queries'
3. 'Thesis: Multiple Kernel Learning for Object Categorization'

These papers have been summarized as follows:

1. 'Pareto-depth for Multiple-query Image Retrieval': Most content-based image retrieval systems consider either one single query, or multiple queries that include the same object or represent the same semantic information. We propose a novel multiple-query information retrieval algorithm that combines the Pareto front method (PFM) with efficient manifold ranking (EMR).
2. 'Building a Large-scale Multimodal Knowledge Base System for Answering Visual Queries': Today's vision systems would still struggle to deal with visual queries that require a deeper reasoning. We propose a knowledge base (KB) framework to handle an assortment of visual queries. Building such a large-scale multimod

In [134]:


@tool
def search_and_summarize(query: str, k: int = 5):
    """
    Search research papers from the FAISS database,
    retrieve the top-k most similar papers,
    summarize each paper using BART,
    and return the results as structured data.
    """

    query_embedding = model.encode([query])

    faiss.normalize_L2(query_embedding)


    D, I = index.search(query_embedding, k)

    papers = []

    for rank, (score, idx) in enumerate(zip(D[0], I[0]), start=1):

        paper = df.iloc[idx]

        summary = summarizer(
            paper["abstract"],
            max_length=120,
            min_length=40,
            do_sample=False
        )[0]["summary_text"]


        paper_data = {
            "rank": rank,
            "similarity": round(float(score), 4),
            "title": paper["title"],
            "abstract": paper["abstract"],
            "summary": summary
        }

        papers.append(paper_data)

    return papers

In [135]:


@tool
def extract_keywords(text: str, top_n: int = 5) -> str:
    """
    Extract the most important keywords from the given text
    using the KeyBERT model.
    """


    keywords = kw_model.extract_keywords(
        text,
        keyphrase_ngram_range=(1, 2),
        stop_words="english",
        top_n=top_n
    )

    result = "Top Keywords:\n\n"

    for rank, (keyword, score) in enumerate(keywords, start=1):

        result += (
            f"{rank}. {keyword} "
            f"(Relevance Score: {round(score, 4)})\n"
        )

    return result

In [136]:
tools = [
    search_and_summarize,
    extract_keywords
]

In [138]:
agent_runnable = create_react_agent(llm, tools, prompt)
agent = AgentExecutor(agent=agent_runnable, tools=tools, verbose=True)

In [ ]:
print(tools)

In [142]:
response = agent.invoke(
    {
        "input": "Extract the top 5 keywords from Deep Learning for Medical Image Reconstruction."
    }
)



> Entering new AgentExecutor chain...
Thought: I need to extract keywords from a given text.
Action: extract_keywords
Action Input: text="Deep Learning for Medical Image Reconstruction"Top Keywords:

1. text deep (Relevance Score: 0.5696)
2. medical image (Relevance Score: 0.5412)
3. deep learning (Relevance Score: 0.4964)
4. text (Relevance Score: 0.4472)
5. image reconstruction (Relevance Score: 0.4425)
Thought: The top 5 keywords for the given text have been successfully extracted.
Final Answer: The top 5 keywords are: text deep, medical image, deep learning, text, and image reconstruction.

> Finished chain.


In [143]:
response = extract_keywords.invoke(
    {
        "text": "Deep Learning for Medical Image Reconstruction",
        "top_n": 5
    }
)

print(response)

Top Keywords:

1. image reconstruction (Relevance Score: 0.5655)
2. deep learning (Relevance Score: 0.526)
3. medical image (Relevance Score: 0.4839)
4. learning medical (Relevance Score: 0.4024)
5. reconstruction (Relevance Score: 0.3745)



In [144]:

user_query = "Extract the top 5 keywords from Deep Learning for Medical Image Reconstruction."

llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke(user_query)

print(response)



print(response.tool_calls)




tool_call = response.tool_calls[0]

print(tool_call)

content='' additional_kwargs={'tool_calls': [{'id': 'ex2v6hj77', 'function': {'arguments': '{"text":"Deep Learning for Medical Image Reconstruction","top_n":5}', 'name': 'extract_keywords'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 588, 'total_tokens': 614, 'completion_time': 0.035649983, 'completion_tokens_details': None, 'prompt_time': 0.0657794, 'prompt_tokens_details': None, 'queue_time': 0.06084814, 'total_time': 0.101429383}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None} id='run--019f7939-1271-7712-8fa8-43e1d6ae5366-0' tool_calls=[{'name': 'extract_keywords', 'args': {'text': 'Deep Learning for Medical Image Reconstruction', 'top_n': 5}, 'id': 'ex2v6hj77', 'type': 'tool_call'}] usage_metadata={'input_tokens': 588, 'output_tokens': 26, 'total_tokens': 614}
[{'name': 'extract_keywords', 'args': {'text': 'Deep Learning

In [145]:
tool_name = tool_call["name"]

print(tool_name)

extract_keywords


In [146]:
tool_args = tool_call["args"]

print(tool_args)

{'text': 'Deep Learning for Medical Image Reconstruction', 'top_n': 5}


In [147]:
if tool_name == "extract_keywords":

    tool_result = extract_keywords.invoke(tool_args)

elif tool_name == "search_and_summarize":

    tool_result = search_and_summarize.invoke(tool_args)

print(tool_result)

Top Keywords:

1. image reconstruction (Relevance Score: 0.5655)
2. deep learning (Relevance Score: 0.526)
3. medical image (Relevance Score: 0.4839)
4. learning medical (Relevance Score: 0.4024)
5. reconstruction (Relevance Score: 0.3745)



In [148]:
from langchain_core.messages import ToolMessage

In [149]:
tool_message = ToolMessage(
    content=tool_result,
    tool_call_id=tool_call["id"]
)

In [150]:
print(tool_message)

content='Top Keywords:\n\n1. image reconstruction (Relevance Score: 0.5655)\n2. deep learning (Relevance Score: 0.526)\n3. medical image (Relevance Score: 0.4839)\n4. learning medical (Relevance Score: 0.4024)\n5. reconstruction (Relevance Score: 0.3745)\n' tool_call_id='ex2v6hj77'


In [151]:
from langchain_core.messages import SystemMessage, HumanMessage

final_response = llm.invoke(
    [
        SystemMessage(
            content="""
You are a helpful AI assistant.

Rules:
1. Always use the tool output.
2. Never ignore tool results.
3. Present the complete tool output.
4. Add a short explanation after the tool output if necessary.
"""
        ),

        HumanMessage(content=user_query),

        response,

        tool_message
    ]
)

print(final_response.content)

The top 5 keywords extracted from the given text are related to the field of medical image reconstruction using deep learning techniques. These keywords can be useful for search engine optimization (SEO), topic modeling, or information retrieval tasks.


In [152]:
from langchain_core.messages import (
    SystemMessage,
    HumanMessage
)

final_response = llm.invoke(
[
SystemMessage(
content="""
You are a research assistant.

IMPORTANT RULES

1. Never ignore tool output.

2. Always display every keyword returned by the tool.

3. Do not summarize the keywords.

4. Print the keywords exactly as received.

5. After printing them, give a short explanation.
"""
),

HumanMessage(content=user_query),

response,

tool_message

]
)

print(final_response.content)

Explanation: The top 5 keywords extracted from the given topic are related to the application of deep learning techniques in medical image reconstruction. These keywords highlight the importance of using deep learning algorithms to improve the quality and accuracy of medical images, which is a crucial aspect of medical imaging.


In [153]:
from langchain_core.tools import tool

@tool
def compare_papers(
    comparison_query: str
) -> str:
    """
    Compares two research papers based on their abstracts.
    The `comparison_query` string should contain the titles of the two papers to compare,
    separated by ' vs ' or ' and '. The tool will extract these titles,
    retrieve the closest matching papers from the FAISS database,
    and use the LLM to compare them.

    Example: 'Compare paper "Title A" vs paper "Title B"'
    """

    if ' vs ' in comparison_query:
        parts = comparison_query.split(' vs ', 1)
    elif ' and ' in comparison_query:
        parts = comparison_query.split(' and ', 1)
    else:
        raise ValueError("comparison_query must contain ' vs ' or ' and ' to separate paper titles.")

    paper1_title = parts[0].replace('paper "', '').replace('"', '').strip()
    paper2_title = parts[1].replace('paper "', '').replace('"', '').strip()

    embedding1 = model.encode([paper1_title])
    faiss.normalize_L2(embedding1)
    D1, I1 = index.search(embedding1, 1)
    first_paper = df.iloc[I1[0][0]]

    embedding2 = model.encode([paper2_title])
    faiss.normalize_L2(embedding2)
    D2, I2 = index.search(embedding2, 1)
    second_paper = df.iloc[I2[0][0]]


    comparison_prompt = f"""
Compare the following two research papers.

Paper 1

Title:
{first_paper['title']}

Abstract:
{first_paper['abstract']}


Paper 2

Title:
{second_paper['title']}

Abstract:
{second_paper['abstract']}


Compare them based on:

1. Research Objective

2. Methodology

3. Key Contributions

4. Advantages

5. Limitations

6. Applications

Present the comparison in a clear table.
"""


    response = llm.invoke(comparison_prompt)

    return response.content

Now that the agent has the `compare_papers` tool, let's try to compare two papers. For example, we can compare the first two papers from our dataset based on their titles.

In [154]:
# Get titles for comparison
paper1_title = df.iloc[0]['title']
paper2_title = df.iloc[1]['title']

print(f"Paper 1: {paper1_title}")
print(f"Paper 2: {paper2_title}")

response = agent.invoke(
    {
        "input": f"Compare paper \"{paper1_title}\" vs paper \"{paper2_title}\"."
    }
)

# The agent's response will contain a list of messages. The final message should have the comparison.
# We'll print the content of the last message.
print("\nAgent's response:")
print(response["output"])

Paper 1: Learning from compressed observations
Paper 2: Sensor Networks with Random Links: Topology Design for Distributed
  Consensus


> Entering new AgentExecutor chain...
Thought: To compare the two papers, I need to use the compare_papers tool.
Action: compare_papers
Action Input: comparison_query='Compare paper "Learning from compressed observations" vs paper "Sensor Networks with Random Links: Topology Design for Distributed Consensus"'**Comparison of Research Papers**

| **Criteria** | **Paper 1: Near-optimal sample compression for nearest neighbors** | **Paper 2: Sensor Networks with Random Links: Topology Design for Distributed Consensus** |
| --- | --- | --- |
| **Research Objective** | Develop a sample compression algorithm for nearest neighbors with non-trivial performance guarantees | Design the topology of a sensor network to maximize the rate of convergence of average consensus under random link failures and communication costs |
| **Methodology** | Algorithmic approach